In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta
import plotly.express as px
from IPython.display import HTML
from IPython.display import Image
import warnings
warnings.filterwarnings('ignore')

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-08-14 18:03:53 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/mmm_tdc/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Introducción

Se consideran como fallas tecnológicas los errores bloqueantes en la landing de conversion, experiencia y en motor.

Los errores bloqueantes son aquellos errores que no permitirian a un usuario con preaprobado obtener su tarjeta de crédito, cuando este tenia todas las condiciones para obtenerlo.

La forma de obtenerlos fue através de la tabla `resultados_vspc_canales.fco_logex_vd`. No hay un filtro que permita obtenerlos, por tanto se obtuvieron manualmente. Primero se generaron los errores únicos desde el año 2024 hasta el 2026 [20260730] teniendo en cuenta las columnas nombre_experiencia, codigo_experiencia, codigo_error, descripcion_tecnica_error, descripcion_funcional_error, servicio_error, operacion_error, tipo_error y categoria_error. El código usado fue, ayudó a construirlo **Jennifer Rivera Cardona**, DISEÑADOR/A PROCESOS - LDC FC TRANSFORMACION DIGITAL [equipo Fernando Jose Arias Mora, DUEÑO/A PRODUCTO - LDC FC MERCADEO DE TECNOLOGIA]:

```sql
SELECT nombre_experiencia,
       codigo_experiencia,
       codigo_error,
       descripcion_tecnica_error,
       descripcion_funcional_error,
       servicio_error,
       operacion_error,
       tipo_error,
       categoria_error,
       count(*) AS cantidad_registros
FROM resultados_vspc_canales.fco_logex_vd
WHERE YEAR BETWEEN 2024 AND 2026
  AND MONTH BETWEEN 1 AND 12
  AND DAY BETWEEN 1 AND 31
  AND cast(codigo_experiencia AS int) IN (10, -- preaprobado mc joven
 11, -- preaprobado mc ideal
 12, -- preaprobado multifranquicia
 21, -- mc negocios segm independientes
 180, -- preaprobado comportamental tdc unica virtual
 111, -- motor mc ideal
 110, -- motor mc joven
 112 -- motor multifranquicia
 )
  AND resultado_trx = 'ERROR'
  AND tipo_error NOT IN ('EXCEPCIONNEGOCIO',
                         'BUSINESS',
                         'NEGOCIO',
                         'EXCEPCIONES DE NEGOCIO',
                         'EXCEPCIONSERVICIO',
                         'BUSINESSEXCEPTIONS')
GROUP BY 1,
         2,
         3,
         4,
         5,
         6,
         7,
         8,
         9
ORDER BY count(*) DESC
```
Luego se generó pasó la tabla generada por la consulta anterior a un archivo excel, *distribucion_frecuencia_errores_experiencia_tdc_20240101_20260804.xlsx*, y con la ayuda de **Karla Cristina Arboleda Alvarez**, DISEÑADOR/A PROCESOS - LDC FC TRANSFORMACION DIGITAL [equipo Fernando Jose Arias Mora, DUEÑO/A PRODUCTO - LDC FC MERCADEO DE TECNOLOGIA], se marcó manualmente cada uno de los errores, según el caso, como error bloqueante.

A medida que pase el tiempo se debe generar nuevamente el archivo y evaluar si hay nuevos registros y marcarlos, según sea el caso, como errores bloqueantes.


In [3]:
# Subir archivo distribucion_frecuencia_errores_experiencia_tdc_20240101_20260804.xlsx a LZ

# Drop table
sql_drop = """
DROP TABLE IF EXISTS proceso.distribucion_frecuencia_errores_experiencia_tdc_20240101_20260804 PURGE;
"""
helper.ejecutar_consulta(sql_drop)

# Leer y modificar columna programables
df_errores_exp_dig = pd.read_excel('hist_data/distribucion_frecuencia_errores_experiencia_tdc_20240101_20260804.xlsx')

# Subir a la lz
sparky.subir_df(df_errores_exp_dig, 'distribucion_frecuencia_errores_experiencia_tdc_20240101_20260804', zona='proceso')

------------------------------------------------------------------------------------------
  i  tipo                  nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 2/2 DROP ...res_experiencia_tdc_20240101_20260804   finalizado   04:39:51 PM     00:01.0 
------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 3/3 DF A LZ ...res_experiencia_tdc_20240101_20260804   ejecutando   04:39:52 PM             

2026-08-14 16:39:58 - [INFO] - Intento 1 de 3


 3/3 DF A LZ ...res_experiencia_tdc_20240101_20260804   finalizado   04:39:52 PM     02:39.5 
---------------------------------------------------------------------------------------------


In [10]:
# Obtener número de errores bloqueantes experiencia digital tdc
sql = """
WITH errores_experiencia_tdc_digital AS
  (SELECT fecha_hora_inicio,
          to_date(fecha_hora_inicio) AS fecha_ymd,
          concat_ws('-', nombre_experiencia, codigo_experiencia, codigo_error, descripcion_tecnica_error, descripcion_funcional_error, servicio_error, operacion_error, tipo_error, cast(categoria_error AS string)) AS llave_error
   FROM resultados_vspc_canales.fco_logex_vd
   WHERE YEAR BETWEEN 2024 AND 2026
     AND MONTH BETWEEN 1 AND 12
     AND DAY BETWEEN 1 AND 31
     AND cast(codigo_experiencia AS int) IN (10, -- preaprobado mc joven
 11, -- preaprobado mc ideal
 12, -- preaprobado multifranquicia
 21, -- mc negocios segm independientes
 180, -- preaprobado comportamental tdc unica virtual
 111, -- motor mc ideal
 110, -- motor mc joven
 112 -- motor multifranquicia
 )
     AND resultado_trx = 'ERROR'
     AND tipo_error NOT IN ('EXCEPCIONNEGOCIO',
                            'BUSINESS',
                            'NEGOCIO',
                            'EXCEPCIONES DE NEGOCIO',
                            'EXCEPCIONSERVICIO',
                            'BUSINESSEXCEPTIONS') ),
     clasificacion_errores AS
  (SELECT nombre_experiencia,
          codigo_experiencia,
          codigo_error,
          descripcion_tecnica_error,
          descripcion_funcional_error,
          servicio_error,
          operacion_error,
          tipo_error,
          categoria_error,
          error_bloqueante,
          concat_ws('-', nombre_experiencia, cast(codigo_experiencia AS string), codigo_error, descripcion_tecnica_error, descripcion_funcional_error, servicio_error, operacion_error, tipo_error, cast(categoria_error AS string)) AS llave_error
   FROM proceso.distribucion_frecuencia_errores_experiencia_tdc_20240101_20260804),
     errores_bloqueantes AS
  (SELECT a.fecha_ymd,
          b.error_bloqueante
   FROM errores_experiencia_tdc_digital AS a
   LEFT JOIN clasificacion_errores AS b ON a.llave_error = b.llave_error
   WHERE b.error_bloqueante = 1 )
SELECT fecha_ymd,
       count(*) AS num_errores_bloqueantes_experiencia_digital
FROM errores_bloqueantes
GROUP BY 1
ORDER BY fecha_ymd
"""
df = helper.obtener_dataframe(sql)

------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 2/2 DATAFRAME         ejecutando   06:07:17 PM             

2026-08-14 18:07:23 - [INFO] - 214 filas, 2 columnas, 00:05.4 consultando, 00:00.0 descargando, 00:00.0 convirtiendo


 2/2 DATAFRAME         finalizado   06:07:17 PM     00:05.5 
------------------------------------------------------------


In [11]:
df

,fecha_ymd,num_errores_bloqueantes_experiencia_digital
0,2026-01-09,113
1,2026-01-10,16
2,2026-01-11,2
3,2026-01-12,14
4,2026-01-13,23
...,...,...
209,2026-08-10,5
210,2026-08-11,8
211,2026-08-12,12
212,2026-08-13,82


In [12]:
# Escribir
df.to_excel('hist_data/errores_bloqueantes_experiencia_digital_tdc_20240101_20260731.xlsx', index=False) # MODIFICAR nombre archivo. Cambia según fecha de ejecución.

# Evolución número de errores bloqueantes

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 2 columns):
 #   Column                                       Non-Null Count  Dtype 
---  ------                                       --------------  ----- 
 0   fecha_ymd                                    214 non-null    object
 1   num_errores_bloqueantes_experiencia_digital  214 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 3.5+ KB


In [9]:
df

,fecha_ymd,num_errores_bloqueantes_experiencia_digital
0,2026-08-14,7
1,2026-08-13,82
2,2026-08-12,12
3,2026-08-11,8
4,2026-08-10,5
...,...,...
209,2026-01-13,23
210,2026-01-12,14
211,2026-01-11,2
212,2026-01-10,16


In [15]:
# Transformar columna fecha_ymd a tipo fecha
df['fecha_ymd'] = pd.to_datetime(df['fecha_ymd'], format='%Y-%m-%d')

fig = px.line(df, x="fecha_ymd", y="num_errores_bloqueantes_experiencia_digital",markers=True, template='simple_white', width=1600, height=800)
# fig.add_vline(x='2025-06-01', line_width=3, line_dash="dash")
# fig.update_layout(
#     title={
#         'text': "Evolución Número de códigos únicos vinculados wompi viejos y totales",
#         'xanchor': 'left',
#         'yanchor': 'top'},
#     xaxis=dict(
#             title=dict(
#                 text="Fecha"
#             )),
#     yaxis=dict(
#             title=dict(
#                 text="Número de códigos únicos vinculados wompi"
#             ))
# )
# fig.add_annotation(x='2025-06-01', y=140000,
#             text="Inicio estrategia 5X",
#             showarrow=False,
#             arrowhead=1)
# fig.add_annotation(x='2025-12-01', y=112321,
#             text="112.321",
#             showarrow=True,
#             arrowhead=1)
fig.show()

# Preguntas

¿Los errores bloqueantes tienen una evolución creible?

¿Por qué sólo hay errores bloqueantes sólo desde enero de 2026?